# 第 1 集：可视化训练结果

本 Notebook 只做一件事：**读取训练产物并绘图**。
- 输入：`loss_history.csv`、`sin_model.pth`（由 `train.ipynb` 生成）
- 输出：`loss_curve.png`、`fit_curve.png`

> 提示：请先运行 `train.ipynb` 生成产物，再运行本 Notebook。

## 1. 导入依赖

In [1]:
import csv
import os

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

## 2. 参数配置

In [2]:
OUTPUT_DIR = "outputs"
NUM_SAMPLES = 1000

print(f"将从 {OUTPUT_DIR}/ 读取训练产物")

将从 outputs/ 读取训练产物


## 3. 绘制训练/验证 loss 曲线

In [3]:
csv_path = os.path.join(OUTPUT_DIR, "loss_history.csv")
epochs, train_losses, val_losses = [], [], []
with open(csv_path, "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        epochs.append(int(row["epoch"]))
        train_losses.append(float(row["train_loss"]))
        val_losses.append(float(row["val_loss"]))

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, label="train loss")
plt.plot(epochs, val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.grid(True)
loss_fig = os.path.join(OUTPUT_DIR, "loss_curve.png")
plt.savefig(loss_fig, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {loss_fig}")

saved: outputs/loss_curve.png


## 4. 加载模型并绘制拟合对照图

In [4]:
class SinPredictor(nn.Module):
    def __init__(self):
        super(SinPredictor, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SinPredictor()
pth_path = os.path.join(OUTPUT_DIR, "sin_model.pth")
model.load_state_dict(torch.load(pth_path, weights_only=True))
model.eval()

x = torch.linspace(0, 2 * np.pi, NUM_SAMPLES).unsqueeze(1)
y_noisy = torch.sin(x) + 0.05 * torch.randn(NUM_SAMPLES, 1)
with torch.no_grad():
    y_pred = model(x)

plt.figure(figsize=(10, 6))
plt.plot(x.flatten().numpy(), y_noisy.flatten().numpy(), label="Noisy sine wave")
plt.plot(x.flatten().numpy(), y_pred.flatten().numpy(), label="Predicted sine wave", color="r")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Original vs Predicted Sine Wave")
plt.legend()
plt.grid(True)
fit_fig = os.path.join(OUTPUT_DIR, "fit_curve.png")
plt.savefig(fit_fig, dpi=150, bbox_inches="tight")
plt.close()
print(f"saved: {fit_fig}")

saved: outputs/fit_curve.png
